In [1]:
import collections
import random
import numpy as np


class GridWorld:
    """A standard 4x4 Gridworld environment."""

    def __init__(self):
        self.rows = 4
        self.cols = 4

        # Terminal states at (0,0) and (3,3)
        self.terminals = [(0, 0), (3, 3)]
        self.actions = ["UP", "DOWN", "LEFT", "RIGHT"]

        self.reset()

    def reset(self):
        """Resets the environment to a random non-terminal state."""

        while True:
            self.state = (
                random.randint(0, 3),
                random.randint(0, 3)
            )

            if self.state not in self.terminals:
                break

        return self.state

    def step(self, action):
        """Moves the agent and returns (next_state, reward, done)."""

        r, c = self.state

        if action == "UP":
            r = max(0, r - 1)
        elif action == "DOWN":
            r = min(3, r + 1)
        elif action == "LEFT":
            c = max(0, c - 1)
        elif action == "RIGHT":
            c = min(3, c + 1)

        self.state = (r, c)
        done = self.state in self.terminals

        # Every step costs -1 reward
        reward = -1

        return self.state, reward, done


def random_policy(state):
    """A uniform random policy selecting any action with 25% probability."""

    return random.choice(["UP", "DOWN", "LEFT", "RIGHT"])


def generate_episode(env, policy):
    """Generates an episode trajectory following the given policy."""

    episode = []
    state = env.reset()
    done = False

    while not done:
        action = policy(state)
        next_state, reward, done = env.step(action)

        episode.append((state, action, reward))
        state = next_state

    return episode


def first_visit_mc_prediction(env, policy, num_episodes, gamma=1.0):
    """Calculates state values V(s) using First-Visit Monte Carlo."""

    # Sum of returns and visit counts for each state
    returns_sum = collections.defaultdict(float)
    returns_count = collections.defaultdict(float)

    # Estimated state-value table
    V = {}

    for _ in range(num_episodes):

        # 1. Generate an episode
        episode = generate_episode(env, policy)

        # 2. Calculate returns backwards
        G = 0
        visited_states = set()

        for state, action, reward in reversed(episode):
            G = gamma * G + reward

            # First visit of this state in the episode
            if state not in visited_states:
                visited_states.add(state)

                returns_sum[state] += G
                returns_count[state] += 1

        # 3. Calculate average returns
        for state in returns_sum:
            V[state] = (
                returns_sum[state] / returns_count[state]
            )

    return V


def print_value_function(V):
    """Prints the 4x4 Gridworld state-value function."""

    grid = np.zeros((4, 4))

    for r in range(4):
        for c in range(4):
            grid[r, c] = round(V.get((r, c), 0.0), 1)

    print(grid)


# Execution
if __name__ == "__main__":

    # Create Gridworld and run MC prediction
    env = GridWorld()

    print("Running First-Visit Monte Carlo Prediction...")

    state_values = first_visit_mc_prediction(
        env,
        random_policy,
        num_episodes=5000
    )

    print("\nEstimated State-Value Function V(s):")
    print_value_function(state_values)

Running First-Visit Monte Carlo Prediction...

Estimated State-Value Function V(s):
[[  0.   -7.7 -10.3 -12.6]
 [ -7.5  -8.6  -9.1 -10.3]
 [-10.4  -9.1  -8.3  -7.6]
 [-12.8 -10.5  -7.8   0. ]]
